# The Price is Right

## Week 8 Order of Play

Day 1: Modal.com and SpecialistAgent  
Day 2: RAG, FrontierAgent, Ensemble Agent  
Day 3: ScannerAgent, MessengerAgent  
Day 4: AutonomousPlannerAgent and DealAgentFramework  
Day 5: The Price Is Right Finale


Today we'll build another piece of the puzzle: a ScanningAgent that looks for promising deals by subscribing to RSS feeds.

In [6]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from agents.deals import ScrapedDeal, DealSelection
import logging
import requests
load_dotenv(override=True)
openai = OpenAI()
# MODEL = 'gpt-5-mini'
# 1. Initialize your client to route through OpenRouter
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ.get("OPENROUTER_API_KEY")
)

# Choose your open-source engine
MODEL = "openai/gpt-oss-20b"  # Or "groq/llama-3.1-8b-instant"

In [2]:
deals = ScrapedDeal.fetch(show_progress=True)

100%|██████████| 3/3 [02:48<00:00, 56.09s/it]


In [3]:
len(deals)

30

In [4]:
deals[10].describe()

'Title: Refurb Dell Latitude Laptop Sale at Dell Refurbished: Extra 40% off + shipping varies\nDetails: Dell Refurbished is offering a range of refurbished Latitude laptops, with prices starting at $215 after promo code "DELLSUMMER40" is applied. The extra 40% off coupon gets some of the lowest prices we\'ve seen this year. Some exclusions apply like Hot Deals. Each purchase includes the same limited hardware warranty Dell offers on new systems. Sale ends June 14, 2026. Shop Now at Dell Refurbished Store\nFeatures: Grades A and B cosmetic conditions available Processors ranging from 10th to 13th Gen Intel Core RAM options: 8GB, 16GB, 32GB, or 64GB Storage: 256GB, 512GB, or 1TB+ SSD Displays from 13.3" to 17" FHD, QHD+, or UHD+ Windows 10 Pro, Windows 11 Pro, or no OS options\nURL: https://www.dealnews.com/Refurb-Dell-Latitude-Laptop-Sale-at-Dell-Refurbished-Extra-40-off-shipping-varies/21839445.html?iref=rss-c39'

### We are going to ask GPT-5-mini to summarize deals and identify their price

In [7]:
SYSTEM_PROMPT = """You identify and summarize the 5 most detailed deals from a list, by selecting deals that have the most detailed, high quality description and the most clear price.
Respond strictly in JSON with no explanation, using this format. You should provide the price as a number derived from the description. If the price of a deal isn't clear, do not include that deal in your response.
Most important is that you respond with the 5 deals that have the most detailed product description with price. It's not important to mention the terms of the deal; most important is a thorough description of the product.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 
"""

USER_PROMPT_PREFIX = """Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price that is greater than 0.
You should rephrase the description to be a summary of the product itself, not the terms of the deal.
Remember to respond with a short paragraph of text in the product_description field for each of the 5 items that you select.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

Deals:

"""

USER_PROMPT_SUFFIX = "\n\nInclude exactly 5 deals, no more."

In [8]:
# this makes a suitable user prompt given scraped deals

def make_user_prompt(scraped):
    user_prompt = USER_PROMPT_PREFIX
    user_prompt += '\n\n'.join([scrape.describe() for scrape in scraped])
    user_prompt += USER_PROMPT_SUFFIX
    return user_prompt

In [9]:
# Let's create a user prompt for the deals we just scraped, and look at how it begins

user_prompt = make_user_prompt(deals)
print(user_prompt[:2000])
messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": user_prompt}]

Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price that is greater than 0.
You should rephrase the description to be a summary of the product itself, not the terms of the deal.
Remember to respond with a short paragraph of text in the product_description field for each of the 5 items that you select.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

Deals:

Title: ESR 25W 3-in-1 Wireless Charger w/ MagSafe for $26 + free shipping
Details: Apply promo code "ESR2C571US85" to drop this ESR 3-in-1 MagSafe charger to $26, down from its regular price of $140. That beats the Amazon price today by almsot $20. The charger includes Apple-certified 15W MagSafe charging for iPhone, 5W fast charging for Apple Watch, and a built-in cooling fan that ke

In [10]:
# response = openai.chat.completions.parse(model=MODEL, messages=messages, response_format=DealSelection, reasoning_effort="minimal")
# results = response.choices[0].message.parsed
# results

import json
# 3. Call the standard API but inject your Pydantic layout via model_json_schema()
response = client.chat.completions.create(
    model=MODEL,
    messages=messages,
    temperature=0.1,
    
    # 🌟 THIS IS THE SECRET SAUCE FOR OPEN-SOURCE STRUCTURED DATA:
    response_format={
        "type": "json_schema",
        "json_schema": {
            "name": "DealSelectionSchema",
            "strict": True,  # Enforces that the model cannot add hallucinated fields
            "schema": DealSelection.model_json_schema()  # Converts Pydantic directly to JSON Schema
        }
    },
)
# 4. Extract and safely reconstruct into your Pydantic object!
raw_json_string = response.choices[0].message.content

# Turn the string into a dictionary, then instantiate it directly into your class wrapper
parsed_dict = json.loads(raw_json_string)
results = DealSelection(**parsed_dict)

# Now results is a fully validated, type-safe Pydantic list of your 5 deals!
print(f"Successfully processed {len(results.deals)} deals.")
print(results)
# print(results.deals[0].product_description)
# print(results.deals[0].price)

Successfully processed 5 deals.
deals=[Deal(product_description='A compact charging station that delivers 15W MagSafe power to iPhones, 5W fast charge for Apple Watch, and a built‑in cooling fan that keeps the phone below 98°F. It comes with a 33W adapter and a 5‑ft USB‑C cable.', price=26.0, url='https://www.dealnews.com/products/ESR/ESR-25-W-3-in-1-Wireless-Charger-w-Mag-Safe/499109.html?iref=rss-c142'), Deal(product_description='A fold‑able, vacuum‑based mount that locks phones up to 78\u202flb and offers 360° rotation with a 200% stronger MagSafe magnet, ensuring a secure, hands‑free navigation experience on any Tesla dashboard or windshield.', price=12.0, url='https://www.dealnews.com/Meifigno-360-Rotating-Mag-Safe-Car-Mount-for-Tesla-for-12-free-shipping-w-Prime/21839401.html?iref=rss-c142'), Deal(product_description='A 32‑inch curved VA panel with 2560×1440 resolution, 165\u202fHz refresh rate, and 1\u202fms response time, delivering smooth gaming and immersive viewing, plus a 1

In [11]:
for deal in results.deals:
    print(deal.product_description)
    print(deal.price)
    print(deal.url)
    print()

A compact charging station that delivers 15W MagSafe power to iPhones, 5W fast charge for Apple Watch, and a built‑in cooling fan that keeps the phone below 98°F. It comes with a 33W adapter and a 5‑ft USB‑C cable.
26.0
https://www.dealnews.com/products/ESR/ESR-25-W-3-in-1-Wireless-Charger-w-Mag-Safe/499109.html?iref=rss-c142

A fold‑able, vacuum‑based mount that locks phones up to 78 lb and offers 360° rotation with a 200% stronger MagSafe magnet, ensuring a secure, hands‑free navigation experience on any Tesla dashboard or windshield.
12.0
https://www.dealnews.com/Meifigno-360-Rotating-Mag-Safe-Car-Mount-for-Tesla-for-12-free-shipping-w-Prime/21839401.html?iref=rss-c142

A 32‑inch curved VA panel with 2560×1440 resolution, 165 Hz refresh rate, and 1 ms response time, delivering smooth gaming and immersive viewing, plus a 1‑year Allstate warranty.
133.0
https://www.dealnews.com/products/Samsung/Open-Box-Samsung-Odyssey-G5-32-QHD-Curved-Monitor/499099.html?iref=rss-c39

A dual‑function

In [12]:
from agents.scanner_agent import ScannerAgent

In [13]:
agent = ScannerAgent()
result = agent.scan()

In [14]:
result

DealSelection(deals=[Deal(product_description='A 3‑in‑1 wireless charger that supports Apple’s MagSafe for iPhone, Apple Watch, and AirPods, featuring a 15W MagSafe output, a 5W fast charge for the Watch, and a built‑in cooling fan that keeps the phone below 98°F during use. It comes with a 33W power adapter and a 5‑ft USB‑C cable.', price=26.0, url='https://www.dealnews.com/products/ESR/ESR-25-W-3-in-1-Wireless-Charger-w-Mag-Safe/499109.html?iref=rss-c142'), Deal(product_description='A 360° rotating MagSafe car mount designed for Tesla vehicles, offering a fold‑able arm and a vacuum‑based suction base that holds up to 78\u202flb. The mount aligns with the MagSafe magnet to secure phones on rough roads and can be placed on dashboards, windshields, or other surfaces.', price=12.0, url='https://www.deal...')])

### Introducing Pushover

Pushover is a nifty tool for sending Push Notifications to your phone.

It's super easy to set up and install!

Simply visit https://pushover.net/ and click 'Login or Signup' on the top right to sign up for a free account, and create your API keys.

Once you've signed up, on the home screen, click "Create an Application/API Token", and give it any name (like AIEngineer) and click Create Application.

Then add 2 lines to your `.env` file:

PUSHOVER_USER=_put the key that's on the top right of your Pushover home screen and probably starts with a u_  
PUSHOVER_TOKEN=_put the key when you click into your new application called Agents (or whatever) and probably starts with an a_

Remember to save your `.env` file, and run `load_dotenv(override=True)` after saving, to set your environment variables.

Finally, click "Add Phone, Tablet or Desktop" to install on your phone.

In [15]:
load_dotenv(override=True)

True

In [16]:
pushover_user = os.getenv('PUSHOVER_USER')
pushover_token = os.getenv('PUSHOVER_TOKEN')
pushover_url = "https://api.pushover.net/1/messages.json"

In [17]:
def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [18]:
push("MASSIVE DEAL!!")

Push: MASSIVE DEAL!!


In [19]:
from agents.messaging_agent import MessagingAgent

agent = MessagingAgent()
agent.push("SUCH A MASSIVE DEAL!!")

In [20]:
agent.notify("A special deal on Sumsung 60 inch LED TV going at a great bargain", 300, 1000, "www.samsung.com")

c:\Users\HP\Udemy\Ai\venv\Lib\site-packages\pydantic\main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected 9 fields but got 5: Expected `Message` - serialized value may not be as expected [field_name='choices', input_value=Message(content='"BIG SAV...er_specific_fields=None), input_type=Message])
  PydanticSerializationUnexpectedValue(Expected `StreamingChoices` - serialized value may not be as expected [field_name='choices', input_value=Choices(finish_reason='st...r_specific_fields=None)), input_type=Choices])
  return self.__pydantic_serializer__.to_python(
